# ACD Video Worker - Kaggle Runner

This notebook runs the ACD Video Worker pipeline on Kaggle.

## Prerequisites

Set the following Kaggle Secrets:

| Secret | Description |
|--------|-------------|
| `GITHUB_TOKEN` | GitHub personal access token with repo scope |
| `PRIVATE_REPO_URL` | URL to your private acd-video-worker repo |
| `LLM_API_KEY` | API key for OpenAI-compatible LLM endpoint |
| `LLM_BASE_URL` | Base URL for LLM API (e.g. https://api.openai.com/v1) |
| `LLM_MODEL` | Model name (e.g. gpt-4o) |
| `DISCORD_WEBHOOK_URL` | Discord webhook URL for status updates |

Optional secrets:
- `YOUTUBE_COOKIES_FILE`: base64-encoded YouTube cookies
- `YT_DLP_COOKIES_PATH`: path to cookies file

## How to run

1. Create a Kaggle notebook with GPU or CPU
2. Add all secrets above under "Add-ons > Secrets"
3. Run this notebook

The worker will:
- Clone this repo
- Check environment
- Clone external repos (Hermes-Agent, OpenMontage)
- Install skill system
- Check LLM API
- Run the job pipeline
- Sync memory back to GitHub

In [ ]:
import os
import subprocess
import sys

# Read Kaggle secrets
github_token = os.environ.get("GITHUB_TOKEN", "")
repo_url = os.environ.get("PRIVATE_REPO_URL", "")

if not github_token or not repo_url:
    print("ERROR: GITHUB_TOKEN and PRIVATE_REPO_URL must be set as Kaggle secrets.")
    print("Go to Add-ons > Secrets and add them.")
    sys.exit(1)

# Authenticate and clone
auth_url = repo_url.replace("https://", f"https://{github_token}@")

if not os.path.isdir("acd-video-worker"):
    print("Cloning private repo...")
    subprocess.run(["git", "clone", auth_url, "acd-video-worker"], check=True)
else:
    print("Repo already cloned, pulling latest...")
    subprocess.run(["git", "-C", "acd-video-worker", "pull", "--ff-only"], check=False)

os.chdir("acd-video-worker")
print(f"Working directory: {os.getcwd()}")

In [ ]:
print("Starting bootstrap...")
result = subprocess.run(
    ["bash", "bootstrap/bootstrap_kaggle.sh"],
    capture_output=False,
    text=True
)
print(f"Bootstrap exit code: {result.returncode}")

In [ ]:
print("Worker run complete.")
print("Check state/runs/ for detailed reports.")